# 🚀 Hybrid Ensemble Model for Plant Disease Detection

This notebook implements a production-quality hybrid ensemble using:
1. **EfficientNet-B4** (Fine-grained features)
2. **ResNet50** (Strong convolutional base)
3. **ViT-B/16** (Global context understanding)

Includes: Mixed precision, cosine annealing, and multiple ensemble strategies (Soft Voting, Weighted, Meta-Classifier Stack).

In [ ]:
import os
import time
import copy
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torch.cuda.amp import GradScaler, autocast

import torchvision
from torchvision import datasets, transforms
import timm

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

warnings.filterwarnings("ignore")

# ==============================================================================
# 1. CONFIGURATION
# ==============================================================================
class CFG:
    DATA_ROOT = Path("/kaggle/input/plantvillage-dataset")
    SAVE_DIR  = Path("/kaggle/working/checkpoints")
    
    # Image setting compatible with ResNet50, EfficientNet-B4, and ViT-B/16
    IMAGE_SIZE = 224
    BATCH_SIZE = 64
    NUM_WORKERS = 4
    
    # Training
    EPOCHS = 15
    PATIENCE = 4
    LEARNING_RATE = 1e-4
    WEIGHT_DECAY = 1e-4
    VAL_SPLIT = 0.15
    
    # Device
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    USE_AMP = torch.cuda.is_available()

# Create checkpoint directory
CFG.SAVE_DIR.mkdir(parents=True, exist_ok=True)

# ImageNet statistics
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# ==============================================================================
# 2. DATA AUGMENTATION & LOADING
# ==============================================================================
def get_transforms():
    """Returns training and validation transforms."""
    train_transforms = transforms.Compose([
        transforms.Resize((CFG.IMAGE_SIZE + 32, CFG.IMAGE_SIZE + 32)),
        transforms.RandomResizedCrop(CFG.IMAGE_SIZE, scale=(0.8, 1.0)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.3),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
        transforms.RandomRotation(degrees=20),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])

    val_transforms = transforms.Compose([
        transforms.Resize((CFG.IMAGE_SIZE, CFG.IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])
    
    return train_transforms, val_transforms


class TransformSubset(torch.utils.data.Dataset):
    """Wrapper to apply specific transforms to a subset."""
    def __init__(self, dataset, indices, transform=None):
        self.dataset = dataset
        self.indices = indices
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        img, label = self.dataset[self.indices[idx]]
        if self.transform:
            img = self.transform(img)
        return img, label


def prepare_data():
    """Discovers images, splits them stratifically, and builds DataLoaders."""
    # Find active root folder with class subdirectories
    img_root = CFG.DATA_ROOT
    for dirpath, dirnames, filenames in os.walk(CFG.DATA_ROOT):
        if dirnames:
            sub = Path(dirpath) / dirnames[0]
            if any(f.endswith('.jpg') or f.endswith('.png') for f in os.listdir(sub)):
                img_root = Path(dirpath)
                break
                
    full_dataset = datasets.ImageFolder(root=str(img_root))
    full_dataset.transform = None 
    
    class_names = full_dataset.classes
    num_classes = len(class_names)
    
    # Stratified Train/Val split
    targets = full_dataset.targets
    train_idx, val_idx = train_test_split(
        np.arange(len(targets)), test_size=CFG.VAL_SPLIT, stratify=targets, random_state=42
    )
    
    train_tf, val_tf = get_transforms()
    train_ds = TransformSubset(full_dataset, train_idx, transform=train_tf)
    val_ds   = TransformSubset(full_dataset, val_idx,   transform=val_tf)
    
    train_loader = DataLoader(train_ds, batch_size=CFG.BATCH_SIZE, shuffle=True, 
                              num_workers=CFG.NUM_WORKERS, pin_memory=True, drop_last=True)
    val_loader   = DataLoader(val_ds, batch_size=CFG.BATCH_SIZE, shuffle=False, 
                              num_workers=CFG.NUM_WORKERS, pin_memory=True)
    
    return train_loader, val_loader, class_names, num_classes, val_ds

# ==============================================================================
# 3. MODEL ARCHITECTURES
# ==============================================================================
def get_model(model_name: str, num_classes: int) -> nn.Module:
    """Builds and prepares a pretrained model for specified num_classes."""
    
    if model_name == "efficientnet_b4":
        # using timm for EfficientNet
        model = timm.create_model("efficientnet_b4", pretrained=True, num_classes=num_classes)
        # Freeze stem and earlier stages initially for fine tuning
        for param in model.parameters():
            param.requires_grad = False
        # Unfreeze classifier and final blocks
        for param in model.conv_head.parameters(): param.requires_grad = True
        for param in model.bn2.parameters(): param.requires_grad = True
        for param in model.classifier.parameters(): param.requires_grad = True
            
    elif model_name == "resnet50":
        # using torchvision for ResNet50
        model = torchvision.models.resnet50(weights=torchvision.models.ResNet50_Weights.IMAGENET1K_V2)
        for param in model.parameters():
            param.requires_grad = False
        # Replace classifier
        in_features = model.fc.in_features
        model.fc = nn.Sequential(
            nn.Dropout(p=0.4),
            nn.Linear(in_features, 512),
            nn.ReLU(),
            nn.Dropout(p=0.2),
            nn.Linear(512, num_classes)
        )
        
    elif model_name == "vit_b_16":
        # using timm for ViT-B/16
        model = timm.create_model("vit_base_patch16_224", pretrained=True, num_classes=num_classes)
        for param in model.parameters():
            param.requires_grad = False
        # Unfreeze head
        for param in model.head.parameters(): 
            param.requires_grad = True

    else:
        raise ValueError(f"Unknown model name: {model_name}")
        
    return model

# ==============================================================================
# 4. TRAINING INFRASTRUCTURE
# ==============================================================================
class EarlyStopping:
    def __init__(self, patience=5, min_delta=1e-4, path="best.pth"):
        self.patience = patience
        self.min_delta = min_delta
        self.path = path
        self.counter = 0
        self.best_loss = None
        self.early_stop = False

    def __call__(self, val_loss, model):
        if self.best_loss is None or val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
            torch.save(model.state_dict(), self.path)
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True


def train_one_epoch(model, loader, criterion, optimizer, scaler, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    
    for images, labels in loader:
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        
        with autocast(enabled=CFG.USE_AMP):
            outputs = model(images)
            loss = criterion(outputs, labels)
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        running_loss += loss.item() * images.size(0)
        _, preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()
        total += labels.size(0)
        
    return running_loss / total, correct / total


@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels, all_logits = [], [], []
    
    for images, labels in loader:
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        
        with autocast(enabled=CFG.USE_AMP):
            outputs = model(images)
            loss = criterion(outputs, labels)
            
        running_loss += loss.item() * images.size(0)
        _, preds = outputs.max(1)
        
        correct += preds.eq(labels).sum().item()
        total += labels.size(0)
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_logits.append(outputs.cpu().numpy())
        
    return running_loss/total, correct/total, np.array(all_preds), np.array(all_labels), np.vstack(all_logits)


def fit_model(model_name, num_classes, train_loader, val_loader):
    print(f"\n{'='*50}\nTraining {model_name.upper()}\n{'='*50}")
    
    model = get_model(model_name, num_classes).to(CFG.DEVICE)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    
    # Only train parameters that require gradients
    optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), 
                           lr=CFG.LEARNING_RATE, weight_decay=CFG.WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG.EPOCHS, eta_min=1e-6)
    scaler = GradScaler(enabled=CFG.USE_AMP)
    
    save_path = CFG.SAVE_DIR / f"{model_name}_best.pth"
    early_stop = EarlyStopping(patience=CFG.PATIENCE, path=str(save_path))
    
    for epoch in range(1, CFG.EPOCHS + 1):
        t0 = time.time()
        
        # Unfreeze all layers halfway through for full fine-tuning
        if epoch == CFG.EPOCHS // 2:
            print(f"  [Epoch {epoch}] Unfreezing entire network for fine-tuning...")
            for param in model.parameters(): param.requires_grad = True
            # Rebuild optimizer
            optimizer = optim.AdamW(model.parameters(), lr=CFG.LEARNING_RATE/10, weight_decay=CFG.WEIGHT_DECAY)
            scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG.EPOCHS//2, eta_min=1e-6)

        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, scaler, CFG.DEVICE)
        val_loss, val_acc, _, _, _ = validate(model, val_loader, criterion, CFG.DEVICE)
        scheduler.step()
        
        print(f"Epoch {epoch:02d}/{CFG.EPOCHS} | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} | Time: {time.time()-t0:.1f}s")
        
        early_stop(val_loss, model)
        if early_stop.early_stop:
            print("  -> Early stopping triggered.")
            break
            
    # Load best weights into memory before returning
    model.load_state_dict(torch.load(save_path))
    return model

# ==============================================================================
# 5. HYBRID ENSEMBLE SYSTEM
# ==============================================================================
def get_model_predictions(models_dict, val_loader):
    """Gathers raw logits from all models to be used in ensemble strategies."""
    criterion = nn.CrossEntropyLoss()
    all_logits = {}
    true_labels = None
    
    for name, model in models_dict.items():
        print(f"Gathering predictions for {name}...")
        _, _, _, labels, logits = validate(model, val_loader, criterion, CFG.DEVICE)
        all_logits[name] = logits
        if true_labels is None:
            true_labels = labels
            
    return all_logits, true_labels

def evaluate_predictions(preds, labels, class_names, title="Ensemble"):
    """Computes comprehensive metrics and highlights weak classes."""
    acc = accuracy_score(labels, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(labels, preds, average='macro')
    
    print(f"\n--- {title} Results ---")
    print(f"Accuracy : {acc*100:.2f}%")
    print(f"Precision: {prec:.4f} | Recall: {rec:.4f} | F1: {f1:.4f}")
    
    # Classification report (dict)
    report = classification_report(labels, preds, target_names=class_names, output_dict=True)
    
    # Highlight weakest recall class to satisfy reduction of false negatives
    class_recalls = {c: report[c]['recall'] for c in class_names}
    weakest_class = min(class_recalls, key=class_recalls.get)
    print(f"Weakest Class (Lowest Recall): {weakest_class} ({class_recalls[weakest_class]*100:.2f}%)\n")

# --- Ensemble Approach 1: Soft Voting ---
def soft_voting_ensemble(logits_dict, labels, class_names):
    # Convert logits to probabilities
    probs = [torch.softmax(torch.tensor(l), dim=-1).numpy() for l in logits_dict.values()]
    
    # Average across all 3 models
    avg_probs = np.mean(probs, axis=0)
    final_preds = np.argmax(avg_probs, axis=-1)
    
    evaluate_predictions(final_preds, labels, class_names, "Approach 1: Soft Voting Ensemble")
    return final_preds

# --- Ensemble Approach 2: Weighted Ensemble ---
def weighted_ensemble(logits_dict, labels, class_names, weights=[0.4, 0.2, 0.4]):
    """
    Weights: 
    - EfficientNet (0.4) for fine local textures
    - ResNet50 (0.2) as supporting CNN base
    - ViT (0.4) for strong global receptive field context
    """
    probs = [torch.softmax(torch.tensor(l), dim=-1).numpy() for l in logits_dict.values()]
    
    weighted_probs = np.zeros_like(probs[0])
    for w, p in zip(weights, probs):
        weighted_probs += w * p
        
    final_preds = np.argmax(weighted_probs, axis=-1)
    evaluate_predictions(final_preds, labels, class_names, f"Approach 2: Weighted Ensemble (W={weights})")
    return final_preds

# --- Optional Bonus: Meta-Classifier (Stacking) ---
def stacking_ensemble(logits_dict, labels, class_names, train_indices, val_indices):
    """
    Concatenates logits and trains a fast LogisticRegression Meta-Classifier.
    Requires train/val split inside the validation phase or cross-validated predictions.
    For simplicity, we split the incoming validation set specifically for the meta-learner.
    """
    # Stack logits feature dimension: (N, 3 * NUM_CLASSES)
    X = np.hstack(list(logits_dict.values()))
    y = labels
    
    # Split the validation set further to train the Meta-Classifier
    X_meta_train, X_meta_test, y_meta_train, y_meta_test = train_test_split(
        X, y, test_size=0.5, random_state=42, stratify=y
    )
    
    meta_clf = LogisticRegression(max_iter=1000, multi_class='multinomial')
    meta_clf.fit(X_meta_train, y_meta_train)
    
    preds = meta_clf.predict(X_meta_test)
    evaluate_predictions(preds, y_meta_test, class_names, "Approach 3: Meta-Classifier (Stacking) on Test Half")

# ==============================================================================
# 6. EXECUTION PIPELINE
# ==============================================================================
def main():
    train_loader, val_loader, class_names, num_classes, val_ds = prepare_data()
    
    model_names = ["resnet50", "efficientnet_b4", "vit_b_16"]
    trained_models = {}
    
    # 1. Train Individual Models (or load if already trained)
    for model_name in model_names:
        save_path = CFG.SAVE_DIR / f"{model_name}_best.pth"
        if save_path.exists():
            print(f"\n[INFO] Checkpoint found for {model_name}. Loading directly from disk.")
            m = get_model(model_name, num_classes)
            m.load_state_dict(torch.load(save_path, map_location=CFG.DEVICE))
            m.to(CFG.DEVICE)
            trained_models[model_name] = m
        else:
            trained_models[model_name] = fit_model(model_name, num_classes, train_loader, val_loader)

    # 2. Ensemble Evaluation
    print("\n" + "="*50)
    print("RUNNING HYBRID ENSEMBLE PIPELINE")
    print("="*50)
    
    logits_dict, true_labels = get_model_predictions(trained_models, val_loader)
    
    # Execute Ensembles
    soft_preds = soft_voting_ensemble(logits_dict, true_labels, class_names)
    weight_preds = weighted_ensemble(logits_dict, true_labels, class_names, weights=[0.2, 0.4, 0.4])
    stacking_ensemble(logits_dict, true_labels, class_names, None, None)

if __name__ == "__main__":
    main()
\n